In [1]:
import requests
import pandas as pd
import time

In [2]:
# Headers so Reddit doesn't block us
headers = {"User-Agent": "pstat134project/1.0 (academic research)"}

# The 4 subreddits we are collecting from
subreddits = ["technology", "artificial", "singularity", "MachineLearning"]

# How many posts we want per subreddit
POSTS_PER_SUB = 200

In [3]:
def get_posts(subreddit, limit=200):
    posts = []
    after = None
    
    while len(posts) < limit:
        url = f"https://www.reddit.com/r/{subreddit}/top.json?limit=100&t=year"
        if after:
            url += f"&after={after}"
            
        response = requests.get(url, headers=headers)
        data = response.json()
        
        children = data["data"]["children"]
        if not children:
            break
            
        for post in children:
            p = post["data"]
            posts.append({
                "id":           p["id"],
                "title":        p["title"],
                "selftext":     p["selftext"],
                "upvote_ratio": p["upvote_ratio"],
                "num_comments": p["num_comments"],
                "is_self":      p["is_self"],
                "edited":       p["edited"],
                "created_utc":  p["created_utc"],
                "subreddit":    p["subreddit"]
            })
        
        after = data["data"]["after"]
        if not after:
            break
            
        time.sleep(1)  # be polite to Reddit's servers
    
    return posts[:limit]

In [4]:
all_posts = []

for sub in subreddits:
    print(f"Collecting from r/{sub}...")
    posts = get_posts(sub, POSTS_PER_SUB)
    all_posts.extend(posts)
    print(f"  Got {len(posts)} posts")
    time.sleep(2)

print(f"\nTotal posts collected: {len(all_posts)}")

  Got 200 posts
  Got 200 posts
  Got 200 posts
  Got 200 posts

Total posts collected: 800


In [5]:
# Convert to DataFrame
df = pd.DataFrame(all_posts)

# Preview it
print(df.shape)
print(df["subreddit"].value_counts())
df.head()

(800, 9)
subreddit
technology         200
artificial         200
singularity        200
MachineLearning    200
Name: count, dtype: int64


,id,title,selftext,upvote_ratio,num_comments,is_self,edited,created_utc,subreddit
0,1nimpmu,DOJ Deletes Study Showing Domestic Terrorists ...,,0.88,2681,False,False,1.758041e+09,technology
1,1nlju03,Disney+ cancellation page crashes as customers...,,0.89,3300,False,False,1.758327e+09,technology
2,1nkl8b0,"Yes, Jimmy Kimmel’s suspension was government ...",,0.81,3102,False,False,1.758232e+09,technology
3,1lka83a,Bernie Sanders says that if AI makes us so pro...,,0.94,2067,False,False,1.750869e+09,technology
4,1nttcqd,Disney reportedly lost 1.7 million paid subscr...,,0.91,3252,False,False,1.759178e+09,technology


In [6]:
# Save raw data to CSV
df.to_csv("../data/raw/reddit_raw.csv", index=False)
print("Saved to data/raw/reddit_raw.csv")

Saved to data/raw/reddit_raw.csv


In [8]:
# Check how many posts have empty body text
missing_body = df[df["selftext"].isin(["", "[removed]", "[deleted]"])]
print(f"Posts with missing/removed body text: {len(missing_body)}")
print(f"\nMissing by subreddit:")
print(missing_body["subreddit"].value_counts())

Posts with missing/removed body text: 472

Missing by subreddit:
subreddit
technology         200
artificial         138
singularity        127
MachineLearning      7
Name: count, dtype: int64


## Data Collection Summary

- **Source:** Reddit public JSON API
- **Subreddits:** r/technology, r/artificial, r/singularity, r/MachineLearning
- **Posts per subreddit:** ~200
- **Total posts collected:** ~800
- **Method:** Reddit .json endpoint with pagination
- **Raw data saved to:** data/raw/reddit_raw.csv

### Notes
- Posts with removed or deleted body text will be handled in notebook 02
- Data collected from top posts of the past year